In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

In [28]:
BASE = Path("..")
CORR_DIR = BASE / "correlations"
OUT_DIR = BASE / "skill_luck_decomposition"

LEAGUE_MAP = {
    "bundesliga": "Bundesliga",
    "la_liga": "La Liga",
    "premier_league": "Premier League",
    "serie_a": "Serie A",
}

In [29]:
def canon_league(name: str) -> str:
    raw = str(name).strip()
    key = raw.lower().replace(" ", "_")
    return LEAGUE_MAP.get(key, raw)

In [30]:
# Helper to compute q for vectors (pandas Series)

# Using the formula: actual = q*skill + (1-q)*luck  =>  q = (actual - luck)/(skill - luck)

def compute_q_series(actual, skill, luck):
    denom = skill - luck
    q = (actual - luck) / denom

    # replace the value if denominator = 0 and values tending to infinity
    q = q.replace([np.inf, -np.inf], np.nan)
    
    return q.clip(0, 1)

In [31]:
def read_corr(subfolder: str) -> pd.DataFrame:
    path = CORR_DIR / subfolder / "spearman_first_second_correlations_by_league_season.csv"
    print(f"Reading from: {path}")
    df = pd.read_csv(path)

    if subfolder in ["actual", "pure_skill"]:
        metric_col = "spearman_r"
    elif subfolder in ["pure_luck_result_based", "pure_luck_goals_based"]:
        metric_col = "spearman_r_avg"
    else:
        raise ValueError(f"Unknown subfolder '{subfolder}'")

    if metric_col not in df.columns:
        raise ValueError(
            f"Column '{metric_col}' not found in {path}. "
            f"Columns: {list(df.columns)}"
        )
    
    df["league"] = df["league"].apply(canon_league)
    df["season"] = df["season"].astype(int)

    df = df.rename(columns={metric_col: "rho"})

    if "n_teams" not in df.columns:
        df["n_teams"] = np.nan

    return df[["league", "season", "n_teams", "rho"]]

In [34]:
def build_correlation_q():
    """
    creates a CSV with columns: league, season, n_teams, spearman_r_actual, spearman_r_skill, spearman_r_luck_results_avg, spearman_r_luck_goals_avg, q_results, q_goals
    where: 
    q_results uses the result-based luck simulations, and
    q_goals uses the goals-based luck simulations.
    """

    # get all four variants of the correlations
    df_actual = read_corr("actual")
    df_skill = read_corr("pure_skill")
    df_luck_res = read_corr("pure_luck_result_based")
    df_luck_goals = read_corr("pure_luck_goals_based")

    # match rows by league and season
    key = ["league", "season"]

    # merge everything 
    wide = (
        df_actual.rename(columns={"rho": "spearman_r_actual"})[
            key + ["n_teams", "spearman_r_actual"]
        ]
        .merge(
            df_skill.rename(columns={"rho": "spearman_r_skill"})[
                key + ["spearman_r_skill"]
            ],
            on=key,
            how="inner",
        )
        .merge(
            df_luck_res.rename(columns={"rho": "spearman_r_luck_results_avg"})[
                key + ["spearman_r_luck_results_avg"]
            ],
            on=key,
            how="inner",
        )
        .merge(
            df_luck_goals.rename(columns={"rho": "spearman_r_luck_goals_avg"})[
                key + ["spearman_r_luck_goals_avg"]
            ],
            on=key,
            how="inner",
        )
    )

    # calculate q-values for result-based simulation
    wide["q_results"] = compute_q_series(
        wide["spearman_r_actual"],
        wide["spearman_r_skill"],
        wide["spearman_r_luck_results_avg"],
    )

    # calculate q-values for goals-based simulation
    wide["q_goals"] = compute_q_series(
        wide["spearman_r_actual"],
        wide["spearman_r_skill"],
        wide["spearman_r_luck_goals_avg"],
    )

    # order columns properly
    wide = wide[
        [
            "league",
            "season",
            "n_teams",
            "spearman_r_actual",
            "spearman_r_skill",
            "spearman_r_luck_results_avg",
            "spearman_r_luck_goals_avg",
            "q_results",
            "q_goals",
        ]
    ]

    # output as a csv
    out_path = OUT_DIR / "correlations_skill_luck_decomposition.csv"
    wide.to_csv(out_path, index=False)
    print(f"Saved correlations skill–luck decomposition to: {out_path}")

    return wide

In [35]:
df_out = build_correlation_q()
df_out.head()

Reading from: ../correlations/actual/spearman_first_second_correlations_by_league_season.csv
Reading from: ../correlations/pure_skill/spearman_first_second_correlations_by_league_season.csv
Reading from: ../correlations/pure_luck_result_based/spearman_first_second_correlations_by_league_season.csv
Reading from: ../correlations/pure_luck_goals_based/spearman_first_second_correlations_by_league_season.csv
Saved correlations skill–luck decomposition to: ../skill_luck_decomposition/correlations_skill_luck_decomposition.csv


,league,season,n_teams,spearman_r_actual,spearman_r_skill,spearman_r_luck_results_avg,spearman_r_luck_goals_avg,q_results,q_goals
0,Bundesliga,2004,18,0.446985,1.0,0.047291,0.006225,0.419535,0.443521
1,Bundesliga,2005,18,0.436392,1.0,-0.003703,-0.020696,0.438472,0.447820
2,Bundesliga,2006,18,0.393782,1.0,0.007489,0.114749,0.389208,0.315202
3,Bundesliga,2007,18,0.564385,1.0,-0.015139,0.054009,0.570881,0.539514
4,Bundesliga,2008,18,0.468053,1.0,-0.154854,-0.087889,0.539381,0.511028
